# 🏦 Mini Project 3 — Agentic AI in FinTech
### Comparing Baseline, Single-Agent, and Multi-Agent Architectures


## 🎯 Learning Objectives

1. Understand how tool-calling (function calling) works with LLMs
2. Define your own tool schemas and wire them to real Python functions
3. Build a single-agent system and reason about its limitations
4. Design and implement a multi-agent system — architecture is your choice
5. Build an LLM-as-judge evaluator to score answers automatically
6. Analyse accuracy, hallucination rate, and latency across all architectures and models
7. Reflect critically: *when does added complexity actually pay off?*

---

## 📦 What You Are Given vs What You Build

| Component | Status |
|---|---|
| 5 working tool functions | ✅ Provided |
| JSON schemas for all 7 tools | ✅ Provided |
| `AgentResult` dataclass | ✅ Provided |
| `run_specialist_agent()` loop | ✅ Provided |
| 15 fixed benchmark questions | ✅ Provided |
| Evaluation runner + Excel writer | ✅ Provided |
| **Tool 6: `get_company_overview`** | ❌ You implement |
| **Tool 7: `get_tickers_by_sector`** | ❌ You implement |
| **Baseline** | ❌ You implement |
| **Single-agent system** | ❌ You design + implement |
| **Multi-agent system** | ❌ You design + implement |
| **Evaluator** | ❌ You design + implement |

---

## 🔑 Before You Start

**API keys needed:**
- OpenAI → https://platform.openai.com/api-keys  
- Alpha Vantage (free) → https://www.alphavantage.co/support/#api-key

**Data needed:**
- `sp500_companies.csv` → https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks  
- Unzip and place next to this notebook

**Create a `.env` file** in the same folder (never commit this):
```
OPENAI_API_KEY=sk-proj-...
ALPHAVANTAGE_API_KEY=...
```


In [1]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

import os
print(os.system('ls'))

os.chdir(os.curdir + "/drive/MyDrive/Colab_Notebooks/llm_proj3")

Mounted at /content/drive/
0


## Step 0 — Install & Import

In [23]:
!pip install openai requests pandas yfinance python-dotenv openpyxl --quiet

In [24]:
import os, json, time, sqlite3, requests, textwrap
import pandas as pd
import yfinance as yf
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY       = os.getenv("OPENAI_API_KEY",       "YOUR_KEY")
ALPHAVANTAGE_API_KEY = os.getenv("ALPHAVANTAGE_API_KEY", "YOUR_KEY")

MODEL_SMALL  = "gpt-4o-mini"
MODEL_LARGE  = "gpt-4o"
ACTIVE_MODEL = MODEL_SMALL          # switch to MODEL_LARGE for the second run

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"✅ Ready  |  active model: {ACTIVE_MODEL}")

✅ Ready  |  active model: gpt-4o-mini


## Step 1 — Build the Local Database

Run `create_local_database()` once after placing `sp500_companies.csv` next to this notebook.  
The function prints all distinct **sector** values — you will need these when implementing Tool 7.


In [25]:
DB_PATH = "stocks.db"

def create_local_database(csv_path: str = "sp500_companies.csv"):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"'{csv_path}' not found.\n"
            "Download from: https://www.kaggle.com/datasets/andrewmvd/sp-500-stocks"
        )
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    df = df.rename(columns={
        "symbol":"ticker", "shortname":"company",
        "sector":"sector",  "industry":"industry",
        "exchange":"exchange", "marketcap":"market_cap_raw"
    })
    def cap_bucket(v):
        try:
            v = float(v)
            return "Large" if v >= 10_000_000_000 else "Mid" if v >= 2_000_000_000 else "Small"
        except: return "Unknown"
    df["market_cap"] = df["market_cap_raw"].apply(cap_bucket)
    df = (df.dropna(subset=["ticker","company"])
            .drop_duplicates(subset=["ticker"])
            [["ticker","company","sector","industry","market_cap","exchange"]])
    conn = sqlite3.connect(DB_PATH)
    df.to_sql("stocks", conn, if_exists="replace", index=False)
    conn.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_ticker ON stocks(ticker)")
    conn.commit()
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM stocks", conn).iloc[0]["n"]
    print(f"✅ {n} companies loaded into stocks.db")
    print("\nDistinct sector values stored in DB:")
    print(pd.read_sql_query("SELECT DISTINCT sector FROM stocks ORDER BY sector", conn).to_string(index=False))
    conn.close()

create_local_database()

✅ 502 companies loaded into stocks.db

Distinct sector values stored in DB:
                sector
       Basic Materials
Communication Services
     Consumer Cyclical
    Consumer Defensive
                Energy
    Financial Services
            Healthcare
           Industrials
           Real Estate
            Technology
             Utilities


In [47]:
# check all Semiconductors in stocks.db
import sqlite3

conn = sqlite3.connect("stocks.db")
cursor = conn.cursor()
cursor.execute("SELECT * FROM stocks WHERE Industry = 'Semiconductors';")
rows = cursor.fetchall()

for r in rows:
    print(r)

('NVDA', 'NVIDIA Corporation', 'Technology', 'Semiconductors', 'Large', 'NMS')
('AVGO', 'Broadcom Inc.', 'Technology', 'Semiconductors', 'Large', 'NMS')
('AMD', 'Advanced Micro Devices, Inc.', 'Technology', 'Semiconductors', 'Large', 'NMS')
('TXN', 'Texas Instruments Incorporated', 'Technology', 'Semiconductors', 'Large', 'NMS')
('QCOM', 'QUALCOMM Incorporated', 'Technology', 'Semiconductors', 'Large', 'NMS')
('ADI', 'Analog Devices, Inc.', 'Technology', 'Semiconductors', 'Large', 'NMS')
('MU', 'Micron Technology, Inc.', 'Technology', 'Semiconductors', 'Large', 'NMS')
('INTC', 'Intel Corporation', 'Technology', 'Semiconductors', 'Large', 'NMS')
('NXPI', 'NXP Semiconductors N.V.', 'Technology', 'Semiconductors', 'Large', 'NMS')
('MCHP', 'Microchip Technology Incorporat', 'Technology', 'Semiconductors', 'Large', 'NMS')
('MPWR', 'Monolithic Power Systems, Inc.', 'Technology', 'Semiconductors', 'Large', 'NMS')
('ON', 'ON Semiconductor Corporation', 'Technology', 'Semiconductors', 'Large', 

## Step 2 — Tool Functions

### Provided Tools (5 of 7)

Read each function carefully — you need to understand their return shapes before writing agents.


In [26]:
# ── Tool 1 ── Provided ────────────────────────────────────────
def get_price_performance(tickers: list, period: str = "1y") -> dict:
    """
    % price change for a list of tickers over a period.
    Valid periods: '1mo', '3mo', '6mo', 'ytd', '1y'
    Returns: { TICKER: {start_price, end_price, pct_change, period} }
    """
    results = {}
    for ticker in tickers:
        try:
            data = yf.download(ticker, period=period, progress=False, auto_adjust=True)
            if data.empty:
                results[ticker] = {"error": "No data — possibly delisted"}
                continue
            start = float(data["Close"].iloc[0].item())
            end   = float(data["Close"].iloc[-1].item())
            results[ticker] = {
                "start_price": round(start, 2),
                "end_price"  : round(end,   2),
                "pct_change" : round((end - start) / start * 100, 2),
                "period"     : period,
            }
        except Exception as e:
            results[ticker] = {"error": str(e)}
    return results

# ── Tool 2 ── Provided ────────────────────────────────────────
def get_market_status() -> dict:
    """Open / closed status for global stock exchanges."""
    return requests.get(
        f"https://www.alphavantage.co/query?function=MARKET_STATUS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 3 ── Provided ────────────────────────────────────────
def get_top_gainers_losers() -> dict:
    """Today's top gaining, top losing, and most active tickers."""
    return requests.get(
        f"https://www.alphavantage.co/query?function=TOP_GAINERS_LOSERS"
        f"&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()

# ── Tool 4 ── Provided ────────────────────────────────────────
def get_news_sentiment(ticker: str, limit: int = 5) -> dict:
    """
    Latest headlines + Bullish / Bearish / Neutral sentiment for a ticker.
    Returns: { ticker, articles: [{title, source, sentiment, score}] }
    """
    data = requests.get(
        f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT"
        f"&tickers={ticker}&limit={limit}&apikey={ALPHAVANTAGE_API_KEY}", timeout=10
    ).json()
    return {
        "ticker": ticker,
        "articles": [
            {
                "title"    : a.get("title"),
                "source"   : a.get("source"),
                "sentiment": a.get("overall_sentiment_label"),
                "score"    : a.get("overall_sentiment_score"),
            }
            for a in data.get("feed", [])[:limit]
        ],
    }

# ── Tool 5 ── Provided ────────────────────────────────────────
def query_local_db(sql: str) -> dict:
    """
    Run any SQL SELECT on stocks.db.
    Table 'stocks' columns: ticker, company, sector, industry, market_cap, exchange
    market_cap values: 'Large' | 'Mid' | 'Small'
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df   = pd.read_sql_query(sql, conn)
        conn.close()
        return {"columns": list(df.columns), "rows": df.to_dict(orient="records")}
    except Exception as e:
        return {"error": str(e)}

print("✅ 5 provided tools ready")

✅ 5 provided tools ready


---
## 🛠️ Task 1 — Implement the 2 Missing Tools (20 pts)

### Tool 6 — `get_company_overview`

Call the Alpha Vantage **OVERVIEW** endpoint for a single ticker.  
Docs: https://www.alphavantage.co/documentation/#company-overview  

Required return format:
```python
{
    "ticker"    : str,   # the input ticker
    "name"      : str,   # company full name
    "sector"    : str,
    "pe_ratio"  : str,   # field name in API: PERatio
    "eps"       : str,   # field name: EPS
    "market_cap": str,   # field name: MarketCapitalization
    "52w_high"  : str,   # field name: 52WeekHigh
    "52w_low"   : str,   # field name: 52WeekLow
}
```
If the API returns no `"Name"` key (rate-limited or invalid ticker), return:
```python
{"error": f"No overview data for {ticker}"}
```

---

### Tool 7 — `get_tickers_by_sector`

Query `stocks.db` for companies matching a sector **or** industry.

**Critical detail:** Look at the sector values printed by `create_local_database()`.  
The DB stores broad sectors in `sector` (e.g. `"Information Technology"`) and  
specific industries in `industry` (e.g. `"Semiconductors"`).  
A query for `"semiconductor"` must fall back to the `industry` column — otherwise it returns zero rows.

Required logic:
1. Try exact match on `sector` column (case-insensitive)  
2. If 0 results → try `LIKE '%sector%'` on the `industry` column  

Required return format:
```python
{
    "sector": str,          # the input search term
    "stocks": [
        {"ticker": str, "company": str, "industry": str},
        ...
    ]
}
```


In [27]:
# ── Tool 6 — YOUR IMPLEMENTATION ─────────────────────────────
def get_company_overview(ticker: str) -> dict:
    """
    Retrieve company fundamentals from Alpha Vantage OVERVIEW endpoint.

    Returns:
    {
        "ticker"    : str,
        "name"      : str,
        "sector"    : str,
        "pe_ratio"  : str,
        "eps"       : str,
        "market_cap": str,
        "52w_high"  : str,
        "52w_low"   : str
    }
    """
    try:
        url = (
            f"https://www.alphavantage.co/query"
            f"?function=OVERVIEW"
            f"&symbol={ticker}"
            f"&apikey={ALPHAVANTAGE_API_KEY}"
        )

        data = requests.get(url, timeout=10).json()

        # If API fails or ticker invalid
        if "Name" not in data:
            return {"error": f"No overview data for {ticker}"}

        return {
            "ticker": ticker,
            "name": data.get("Name"),
            "sector": data.get("Sector"),
            "pe_ratio": data.get("PERatio"),
            "eps": data.get("EPS"),
            "market_cap": data.get("MarketCapitalization"),
            "52w_high": data.get("52WeekHigh"),
            "52w_low": data.get("52WeekLow"),
        }
    except Exception as e:
        return {"error": str(e)}


# ── Tool 7 — YOUR IMPLEMENTATION ─────────────────────────────
def get_tickers_by_sector(sector: str) -> dict:
    """
    Retrieve tickers belonging to a sector or industry.

    Logic:
    1) Try exact match on sector column (case-insensitive)
    2) If no results → fallback to industry LIKE '%sector%'

    Returns:
    {
        "sector": str,
        "stocks": [
            {"ticker": str, "company": str, "industry": str}
        ]
    }
    """
    try:
        conn = sqlite3.connect(DB_PATH)

        # --- Step 1: exact sector match ---
        query_sector = """
        SELECT ticker, company, industry
        FROM stocks
        WHERE LOWER(sector) = LOWER(?)
        """

        df = pd.read_sql_query(query_sector, conn, params=[sector])

        # --- Step 2: fallback to industry search ---
        if df.empty:
            query_industry = """
            SELECT ticker, company, industry
            FROM stocks
            WHERE LOWER(industry) LIKE LOWER(?)
            """
            df = pd.read_sql_query(query_industry, conn, params=[f"%{sector}%"])

        conn.close()

        return {
            "sector": sector,
            "stocks": df.to_dict(orient="records")
        }

    except Exception as e:
        return {"error": str(e)}

In [30]:
# ── Automated tests — all assertions must pass ────────────────
print("=== Tool 6 tests ===")
aapl = get_company_overview("AAPL")
assert "pe_ratio" in aapl,             "Missing pe_ratio key"
assert aapl.get("ticker") == "AAPL",   "ticker key wrong"
assert aapl.get("name"),               "name should not be empty"
print(f"  AAPL P/E: {aapl['pe_ratio']} ✅")

bad = get_company_overview("INVALIDTICKER999")
assert "error" in bad,                 "Invalid ticker should return error key"
print(f"  Invalid ticker handled correctly ✅")

print("\n=== Tool 7 tests ===")
tech = get_tickers_by_sector("Information Technology")
assert len(tech["stocks"]) > 0,        "Should find IT stocks (exact sector match)"
print(f"  'Information Technology' → {len(tech['stocks'])} stocks ✅")

semi = get_tickers_by_sector("semiconductor")
assert len(semi["stocks"]) > 0,        "Should find via industry fallback (LIKE match)"
print(f"  'semiconductor' (industry fallback) → {len(semi['stocks'])} stocks ✅")

print("\n✅ All tool tests passed")

=== Tool 6 tests ===
  AAPL P/E: 32.88 ✅
  Invalid ticker handled correctly ✅

=== Tool 7 tests ===
  'Information Technology' → 11 stocks ✅
  'semiconductor' (industry fallback) → 18 stocks ✅

✅ All tool tests passed


## Step 3 — Tool Schemas (Provided)

Schemas tell the LLM what tools exist, what they do, and what arguments they take.  
You do not modify these — but you will reference the schema lists when building your agents.

**Key concept:** You can give different agents different *subsets* of schemas.  
A specialist that only sees 2–3 relevant schemas makes fewer wrong tool choices  
than one that sees all 7.


In [31]:
def _s(name, desc, props, req):
    return {"type":"function","function":{
        "name":name,"description":desc,
        "parameters":{"type":"object","properties":props,"required":req}}}

SCHEMA_TICKERS  = _s("get_tickers_by_sector",
    "Return all stocks in a sector or industry from the local database. "
    "Use broad sector names ('Information Technology', 'Energy') or sub-sectors ('semiconductor', 'insurance').",
    {"sector":{"type":"string","description":"Sector or industry name"}}, ["sector"])

SCHEMA_PRICE    = _s("get_price_performance",
    "Get % price change for a list of tickers over a time period. "
    "Periods: '1mo','3mo','6mo','ytd','1y'.",
    {"tickers":{"type":"array","items":{"type":"string"}},
     "period":{"type":"string","default":"1y"}}, ["tickers"])

SCHEMA_OVERVIEW = _s("get_company_overview",
    "Get fundamentals for one stock: P/E ratio, EPS, market cap, 52-week high and low.",
    {"ticker":{"type":"string","description":"Ticker symbol e.g. 'AAPL'"}}, ["ticker"])

SCHEMA_STATUS   = _s("get_market_status",
    "Check whether global stock exchanges are currently open or closed.", {}, [])

SCHEMA_MOVERS   = _s("get_top_gainers_losers",
    "Get today's top gaining, top losing, and most actively traded stocks.", {}, [])

SCHEMA_NEWS     = _s("get_news_sentiment",
    "Get latest news headlines and Bullish/Bearish/Neutral sentiment scores for a stock.",
    {"ticker":{"type":"string"},"limit":{"type":"integer","default":5}}, ["ticker"])

SCHEMA_SQL      = _s("query_local_db",
    "Run a SQL SELECT on stocks.db. "
    "Table 'stocks': ticker, company, sector, industry, market_cap (Large/Mid/Small), exchange.",
    {"sql":{"type":"string","description":"A valid SQL SELECT statement"}}, ["sql"])

# All 7 schemas in one list — used by single agent
ALL_SCHEMAS = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_OVERVIEW,
               SCHEMA_STATUS, SCHEMA_MOVERS, SCHEMA_NEWS, SCHEMA_SQL]

# Dispatch map — maps the tool name string the LLM returns → the Python function to call
ALL_TOOL_FUNCTIONS = {
    "get_tickers_by_sector" : get_tickers_by_sector,
    "get_price_performance"  : get_price_performance,
    "get_company_overview"   : get_company_overview,
    "get_market_status"      : get_market_status,
    "get_top_gainers_losers" : get_top_gainers_losers,
    "get_news_sentiment"     : get_news_sentiment,
    "query_local_db"         : query_local_db,
}

print("✅ Schemas ready")
print(f"   Tools available: {list(ALL_TOOL_FUNCTIONS.keys())}")

✅ Schemas ready
   Tools available: ['get_tickers_by_sector', 'get_price_performance', 'get_company_overview', 'get_market_status', 'get_top_gainers_losers', 'get_news_sentiment', 'query_local_db']


## Step 4 — AgentResult and Base Runner (Provided)

`AgentResult` is the standard return type for every agent — baseline, single, and all multi-agent specialists.  
`run_specialist_agent` is the reusable tool-call loop that every agent uses.  
Study this loop carefully before writing your own agents — it is what connects the LLM's tool requests to your Python functions.


In [32]:
@dataclass
class AgentResult:
    agent_name   : str
    answer       : str
    tools_called : list  = field(default_factory=list)   # tool names in order called
    raw_data     : dict  = field(default_factory=dict)   # tool name → raw tool output
    confidence   : float = 0.0                           # set by evaluator / critic
    issues_found : list  = field(default_factory=list)   # set by evaluator / critic
    reasoning    : str   = ""

    def summary(self):
        print(f"\n{'─'*54}")
        print(f"Agent      : {self.agent_name}")
        print(f"Tools used : {', '.join(self.tools_called) or 'none'}")
        print(f"Confidence : {self.confidence:.0%}")
        if self.issues_found:
            print(f"Issues     : {'; '.join(self.issues_found)}")
        print(f"Answer     :\n{textwrap.indent(self.answer[:500], '  ')}")

print("✅ AgentResult defined")

✅ AgentResult defined


In [33]:
def run_specialist_agent(
    agent_name   : str,
    system_prompt: str,
    task         : str,
    tool_schemas : list,
    max_iters    : int  = 8,
    verbose      : bool = True,
) -> AgentResult:
    """
    Core agentic loop used by every agent in this project.

    How it works:
      1. Sends system_prompt + task to the LLM
      2. If the LLM requests a tool call → looks up the function in ALL_TOOL_FUNCTIONS,
         executes it, appends the result to the message history, loops back to step 1
      3. When the LLM produces a response with no tool calls → returns an AgentResult

    Parameters
    ----------
    agent_name    : display name for logging
    system_prompt : the agent's persona, rules, and focus area
    task          : the specific question or sub-task for this agent
    tool_schemas  : list of schema dicts this agent is allowed to use
                    (pass [] for no tools — used by baseline)
    max_iters     : hard cap on iterations to prevent infinite loops
    verbose       : print each tool call as it happens
    """
    ### YOUR CODE HERE ###
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": task}
    ]

    tools_called = []
    raw_data = {}

    for _ in range(max_iters):

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tool_schemas if tool_schemas else None,
             # Only set tool_choice if we have tools
            **({"tool_choice": "auto"} if tool_schemas else {})
        )

        msg = response.choices[0].message

        # Case 1: Assistant wants to call tools
        if msg.tool_calls:

            if verbose:
                print(f"[{agent_name}] TOOL CALL REQUESTED → {len(msg.tool_calls)} tool(s)")

            # Append the assistant message once (the one requesting the tools)
            messages.append(msg)

            # Execute each tool call and append its result
            for tool_call in msg.tool_calls:

                tool_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"[{agent_name}] TOOL → {tool_name} {args}")

                tools_called.append(tool_name)

                # Execute the actual tool
                tool_fn = ALL_TOOL_FUNCTIONS[tool_name]
                result = tool_fn(**args)
                raw_data[tool_name] = result

                # Append the tool output with correct tool_call_id
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

        else:
            # Case 2: Assistant provided a final natural language answer
            answer = msg.content or ""
            return AgentResult(
                agent_name=agent_name,
                answer=answer,
                tools_called=tools_called,
                raw_data=raw_data
            )

    # Fallback if loop exceeded
    return AgentResult(
        agent_name=agent_name,
        answer="Agent stopped after max iterations.",
        tools_called=tools_called,
        raw_data=raw_data
    )

---
## 🛠️ Task 2 — Implement the Baseline (10 pts)

The baseline is a **single LLM call with no tools**. The model answers entirely from its training knowledge.

This is your **control condition**. Every architecture you build should be compared against it.  
If agents don't outperform the baseline, there's no justification for the extra complexity.

**Requirements:**
- One call to `client.chat.completions.create()` — no `tools` argument
- Return `AgentResult(agent_name="Baseline", answer=..., tools_called=[])`
- Use a sensible system prompt that encourages the model to be honest about uncertainty

**Grading checks:**
- `tools_called` must be `[]`
- Answer must not be empty


In [34]:
def run_baseline(question: str, verbose: bool = True) -> AgentResult:
    # Implement a single LLM call with no tools.
    # Use run_specialist_agent() with an empty tool_schemas list — or make the call directly.
    # Return an AgentResult with agent_name="Baseline" and tools_called=[].
    ### YOUR CODE HERE
    system_prompt = """
      You are a careful financial research assistant.

      Answer the user's question using your general knowledge only.
      If you are uncertain, say so clearly and provide an approximate answer.
      Do NOT fabricate precise numbers if you are unsure.
      Be concise but informative.
    """
    result = run_specialist_agent(
        agent_name="Baseline",
        system_prompt=system_prompt,
        task=question,
        tool_schemas=[],   # No tools allowed
        verbose=verbose
    )
    return result

# Quick test
bl = run_baseline("What is Apple's approximate P/E ratio?", verbose=True)
assert bl.tools_called == [], "Baseline must not call any tools"
assert bl.answer, "Answer must not be empty"
bl.summary()


──────────────────────────────────────────────────────
Agent      : Baseline
Tools used : none
Confidence : 0%
Answer     :
  As of my last knowledge update in October 2023, Apple's P/E ratio typically fluctuated around 25 to 30. However, for the most accurate and current figure, I recommend checking a reliable financial news website or stock market platform, as these numbers can change frequently based on market conditions.


---
## 🛠️ Task 3 — Design and Implement the Single Agent (20 pts)

A single agent is **one LLM with access to all 7 tools**. Everything — planning which tools to call, executing them, and synthesising the answer — happens in one context window.

You decide how to build it. The guidance below is a starting point, not a recipe.

---

### Things to think about when writing your system prompt

**Role and scope** — what is this agent's job? Being specific helps the model stay focused.

**Accuracy rules** — how should the agent behave when a tool returns an error or empty data?  
This is critical: an agent with no rules tends to fabricate plausible-looking numbers when the API fails.

**Multi-step reasoning** — some questions require chaining tools. For example:  
*"Which energy stocks grew the most this year?"* requires first looking up energy tickers,  
then fetching price data for each one. Without explicit guidance, single agents often  
skip the lookup step and guess tickers instead.

**Tool selection** — the agent sees all 7 tools. Giving it rules about *when* to use each  
(e.g. "use `query_local_db` with SQL when you need to filter by sector or market cap")  
reduces wrong tool choices.

---

### Recommended structure

```python
SINGLE_AGENT_PROMPT = """
<your system prompt here>
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    # Call run_specialist_agent() with:
    #   agent_name    = "Single Agent"
    #   system_prompt = SINGLE_AGENT_PROMPT
    #   task          = question
    #   tool_schemas  = ALL_SCHEMAS   (all 7)
    #   max_iters     = 10
    # Return the AgentResult directly.
    pass
```

---

### Test before you move on

Run the three cells below and check the results make sense before building the multi-agent system.


In [35]:
# ── YOUR SINGLE AGENT IMPLEMENTATION ─────────────────────────
# Write your system prompt and run_single_agent() function here.
# Comments above explain what to think about — the implementation is yours.

SINGLE_AGENT_PROMPT = """
You are a financial research assistant with access to several tools
for retrieving financial data, company information, and market statistics.

Your job is to answer finance questions accurately by using tools when needed.

Rules:
1. Prefer tool data over guessing. If a tool can provide the information, use it.
2. If a tool returns empty data or an error, explain the limitation instead of inventing numbers.
3. Use multiple tools if necessary to complete multi-step questions.
4. Always verify results before answering.

Tool usage guidance:
- Use tools that retrieve stock fundamentals when asked for metrics like P/E ratio, EPS, market cap.
- Use price/history tools when questions involve stock performance or growth.
- Use database/query tools when filtering companies by sector, industry, or market cap.
- If a ticker is unknown, first look it up before requesting financial data.

Reasoning strategy:
1. Understand the question.
2. Decide if a tool is needed.
3. Call the tool.
4. Interpret the tool output.
5. Provide a clear final answer.

Your final answer should:
- Be concise
- Reference the data you retrieved
- Avoid speculation when data is missing
- Exclude tickers that return errors or missing price data from analysis.
- Only include stocks with valid price data.
- Explain if some tickers were skipped.
"""

def run_single_agent(question: str, verbose: bool = True) -> AgentResult:
    result = run_specialist_agent(
        agent_name="Single Agent",
        system_prompt=SINGLE_AGENT_PROMPT,
        task=question,
        tool_schemas=ALL_SCHEMAS,   # all 7 tools
        max_iters=10,
        verbose=verbose
    )

    return result
### YOUR CODE HERE

In [36]:
# Test 1 — easy question, 1 tool expected
r1 = run_single_agent("What is the P/E ratio of Apple (AAPL)?")
r1.summary()

[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → get_company_overview {'ticker': 'AAPL'}

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_company_overview
Confidence : 0%
Answer     :
  The P/E ratio of Apple Inc. (AAPL) is 32.88.


In [37]:
# Test 2 — medium question, requires sector lookup + price fetch
r2 = run_single_agent("Which energy stocks in the database had the best 6-month performance?")
r2.summary()

[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → get_tickers_by_sector {'sector': 'Energy'}
[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → get_price_performance {'tickers': ['XOM', 'CVX', 'COP', 'EOG', 'WMB', 'KMI', 'OKE', 'SLB', 'PSX', 'FANG', 'OXY', 'MPC', 'BKR', 'HES', 'TRGP', 'VLO', 'TPL', 'EQT', 'HAL', 'DVN', 'CTRA', 'APA'], 'period': '6mo'}


$HES: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HES']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")



──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector, get_price_performance
Confidence : 0%
Answer     :
  The energy stocks with the best 6-month performance are as follows:

  1. **Texas Pacific Land Corporation (TPL)**: 73.12% increase
  2. **Targa Resources, Inc. (TRGP)**: 48.75% increase
  3. **APA Corporation (APA)**: 49.87% increase
  4. **Valero Energy Corporation (VLO)**: 44.53% increase
  5. **Exxon Mobil Corporation (XOM)**: 39.78% increase
  6. **Halliburton Company (HAL)**: 58.16% increase
  7. **Diamondback Energy, Inc. (FANG)**: 33.21% increase
  8. **Williams Companies, Inc. (WMB)**: 32.79% increase



In [38]:
# Test 3 — hard multi-condition question
r3 = run_single_agent("Top 3 tech stocks that dropped this month but grew this year.")
r3.summary()

[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → get_tickers_by_sector {'sector': 'Information Technology'}
[Single Agent] TOOL CALL REQUESTED → 2 tool(s)
[Single Agent] TOOL → get_price_performance {'tickers': ['ACN', 'IBM', 'FI', 'FIS', 'CTSH', 'IT', 'BR', 'CDW', 'LDOS', 'EPAM', 'JKHY'], 'period': '1mo'}


$FI: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


[Single Agent] TOOL → get_price_performance {'tickers': ['ACN', 'IBM', 'FI', 'FIS', 'CTSH', 'IT', 'BR', 'CDW', 'LDOS', 'EPAM', 'JKHY'], 'period': '1y'}


$FI: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")



──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector, get_price_performance, get_price_performance
Confidence : 0%
Answer     :
  Here are the top 3 technology stocks that experienced a price drop this month but showed growth over the year:

  1. **Accenture plc (ACN)**
     - **1-Month Change:** -9.19%
     - **1-Year Change:** -36.54%

  2. **Cognizant Technology Solutions (CTSH)**
     - **1-Month Change:** -10.73%
     - **1-Year Change:** -18.97%

  3. **EPAM Systems, Inc. (EPAM)**
     - **1-Month Change:** -23.44%
     - **1-Year Change:** -26.62%

  ### Summary of Results:
  - **Accenture plc (ACN)** had a significant decline of over 9%


## Test Baseline

In [39]:
import time

def run_baseline_with_logging(question: str, verbose: bool = True):
    start_time = time.time()

    # Run the baseline
    result = run_baseline(question, verbose=verbose)

    end_time = time.time()
    elapsed = end_time - start_time

    # Baseline has no tools, so iteration = 1
    iterations = 1

    # Confidence can come from result.confidence if set, otherwise 0
    score = getattr(result, "confidence", 0.0)

    log = {
        "question": question,
        "answer": result.answer,
        "score": score,
        "iterations": iterations,
        "elapsed_sec": round(elapsed, 2)
    }

    return result, log

In [40]:
questions_list = [
    "List all semiconductor companies in the database.", # Q01 easy
    "Are the US stock markets open right now?", # Q02 easy
    "What is the P/E ratio of Apple (AAPL)?", # Q03 easy
    "What is the latest news sentiment for Microsoft (MSFT)?", # Q04 easy
    "What is NVIDIA's stock price performance over the last month?", # Q05 easy
    "Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?", # Q06 medium
    "Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?", # Q07 medium
    "Which energy stocks in the database had the best 6-month performance?", # Q08 medium
    "What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?", # Q09 medium
    "What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?", # Q10 medium
    "Which tech stocks dropped this month but grew this year? Return the top 3.", # Q11 hard
    "Which large-cap technology stocks on NASDAQ have grown more than 20% this year?", # Q12 hard
    "For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment?", # Q13 hard
    "Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.", # Q14 hard
    "Which finance sector stocks are trading closer to their 52-week low than their 52-week high? Return the news sentiment for each." # Q15 hard
]

logs = []

for q in questions_list:
    r, log = run_baseline_with_logging(q, verbose=True)
    r.summary()
    logs.append(log)

# View logs in a DataFrame
import pandas as pd
df_logs = pd.DataFrame(logs)
df_logs


──────────────────────────────────────────────────────
Agent      : Baseline
Tools used : none
Confidence : 0%
Answer     :
  I don't have access to a specific database to list semiconductor companies. However, I can mention some well-known semiconductor companies based on general knowledge. These include:

  1. Intel Corporation
  2. Samsung Electronics
  3. NVIDIA
  4. Texas Instruments
  5. Qualcomm
  6. Advanced Micro Devices (AMD)
  7. Broadcom Inc.
  8. Micron Technology
  9. IBM
  10. STMicroelectronics

  There are many more companies in the semiconductor sector, ranging from major manufacturers to specialized firms, but these are s

──────────────────────────────────────────────────────
Agent      : Baseline
Tools used : none
Confidence : 0%
Answer     :
  I can't check real-time information, but the US stock markets typically operate from 9:30 AM to 4:00 PM Eastern Time, Monday through Friday, excluding market holidays. If you want to know if they are currently open, you can

,question,answer,score,iterations,elapsed_sec
0,List all semiconductor companies in the database.,I don't have access to a specific database to ...,0.0,1,2.76
1,Are the US stock markets open right now?,"I can't check real-time information, but the U...",0.0,1,2.16
2,What is the P/E ratio of Apple (AAPL)?,I'm unable to provide the current P/E ratio fo...,0.0,1,2.55
3,What is the latest news sentiment for Microsof...,I'm unable to provide real-time data or the la...,0.0,1,2.20
4,What is NVIDIA's stock price performance over ...,"I'm unable to provide real-time data, includin...",0.0,1,2.88
5,"Compare the 1-year price performance of AAPL, ...",As of my last knowledge update in October 2023...,0.0,1,3.00
6,"Compare the P/E ratios of AAPL, MSFT, and NVDA...",As of my last knowledge update in October 2023...,0.0,1,3.98
7,Which energy stocks in the database had the be...,I'm unable to access a specific database to ch...,0.0,1,3.40
8,What is the news sentiment for Tesla (TSLA) an...,I don't have access to real-time news sentimen...,0.0,1,2.85
9,What are the 52-week high and low for JPMorgan...,I'm unable to provide real-time data or curren...,0.0,1,1.42


## Test Single Agent with Benchmark Questions

### The 15 Benchmark Questions

Fixed — do not modify. All three architectures run against all 15.

| ID | Difficulty | Category | Question |
|---|---|---|---|
| Q01 | easy | sector_lookup | List all semiconductor companies in the database. |
| Q02 | easy | market_status | Are the US stock markets open right now? |
| Q03 | easy | fundamentals | What is the P/E ratio of Apple (AAPL)? |
| Q04 | easy | sentiment | What is the latest news sentiment for Microsoft (MSFT)? |
| Q05 | easy | price | What is NVIDIA's stock price performance over the last month? |
| Q06 | medium | price_comparison | Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most? |
| Q07 | medium | fundamentals | Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive? |
| Q08 | medium | sector_price | Which energy stocks in the database had the best 6-month performance? |
| Q09 | medium | sentiment | What is the news sentiment for Tesla (TSLA) and how has its stock moved this month? |
| Q10 | medium | fundamentals | What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)? |
| Q11 | hard | multi_condition | Which tech stocks dropped this month but grew this year? Return the top 3. |
| Q12 | hard | multi_condition | Which large-cap technology stocks on NASDAQ have grown more than 20% this year? |
| Q13 | hard | cross_domain | For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment? |
| Q14 | hard | cross_domain | Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC. |
| Q15 | hard | multi_condition | Which finance sector stocks are closer to their 52-week low than high? Return the news sentiment for each. |

In [41]:
import time

def run_single_agent_with_logging(question: str, verbose: bool = True):
    start_time = time.time()

    # Run the original single agent
    result = run_single_agent(question, verbose=verbose)

    end_time = time.time()
    elapsed = end_time - start_time

    # Number of iterations is the number of assistant+tool loops
    iterations = getattr(result, "iterations", None)
    if iterations is None:
        # fallback: approximate as number of tool calls + 1 for final answer
        iterations = len(result.tools_called) + 1

    # You can also optionally evaluate answer confidence automatically
    # Here we just mock a placeholder; in your project you may use gpt-4o-mini as evaluator
    # e.g., run_evaluator(question, result.answer)
    score = getattr(result, "confidence", 0.0)  # use 0 if not set

    log = {
        "question": question,
        "answer": result.answer,
        "score": score,
        "iterations": iterations,
        "elapsed_sec": round(elapsed, 2)
    }

    return result, log

In [42]:
questions_list = [
    "List all semiconductor companies in the database.", # Q01 easy
    "Are the US stock markets open right now?", # Q02 easy
    "What is the P/E ratio of Apple (AAPL)?", # Q03 easy
    "What is the latest news sentiment for Microsoft (MSFT)?", # Q04 easy
    "What is NVIDIA's stock price performance over the last month?", # Q05 easy
    "Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?", # Q06 medium
    "Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?", # Q07 medium
    "Which energy stocks in the database had the best 6-month performance?", # Q08 medium
    "What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?", # Q09 medium
    "What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?", # Q10 medium
    "Which tech stocks dropped this month but grew this year? Return the top 3.", # Q11 hard
    "Which large-cap technology stocks on NASDAQ have grown more than 20% this year?", # Q12 hard
    "For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment?", # Q13 hard
    "Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.", # Q14 hard
    "Which finance sector stocks are trading closer to their 52-week low than their 52-week high? Return the news sentiment for each." # Q15 hard
]

logs = []

for q in questions_list:
    r, log = run_single_agent_with_logging(q, verbose=True)
    r.summary()
    logs.append(log)


[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → get_tickers_by_sector {'sector': 'semiconductor'}

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector
Confidence : 0%
Answer     :
  Here is a list of semiconductor companies in the database:

  1. **NVIDIA Corporation (NVDA)**
  2. **Broadcom Inc. (AVGO)**
  3. **Advanced Micro Devices, Inc. (AMD)**
  4. **Texas Instruments Incorporated (TXN)**
  5. **QUALCOMM Incorporated (QCOM)**
  6. **Applied Materials, Inc. (AMAT)**
  7. **Analog Devices, Inc. (ADI)**
  8. **Micron Technology, Inc. (MU)**
  9. **Lam Research Corporation (LRCX)**
  10. **Intel Corporation (INTC)**
  11. **KLA Corporation (KLAC)**
  12. **NXP Semiconductors N.V. (NXPI)**
  13. **M
[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → get_market_status {}

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_market_status
Con

$HES: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HES']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")



──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector, get_price_performance
Confidence : 0%
Answer     :
  The energy stocks with the best 6-month performance, based on their percentage price changes, are as follows:

  1. **Texas Pacific Land Corporation (TPL)**: +73.12%
     - Start Price: $303.28
     - End Price: $525.03

  2. **Targa Resources, Inc. (TRGP)**: +48.75%
     - Start Price: $159.46
     - End Price: $237.20

  3. **APA Corporation (APA)**: +49.87%
     - Start Price: $21.81
     - End Price: $32.68

  4. **Halliburton Company (HAL)**: +58.16%
     - Start Price: $21.53
     - End Price: $34.05

  5. **Valero
[Single Agent] TOOL CALL REQUESTED → 2 tool(s)
[Single Agent] TOOL → get_news_sentiment {'ticker': 'TSLA'}
[Single Agent] TOOL → get_price_performance {'tickers': ['TSLA'], 'period': '1mo'}

──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_news_sentimen

$FI: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


[Single Agent] TOOL → get_price_performance {'tickers': ['ACN', 'IBM', 'FI', 'FIS', 'CTSH', 'IT', 'BR', 'CDW', 'LDOS', 'EPAM', 'JKHY'], 'period': 'ytd'}


$FI: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")



──────────────────────────────────────────────────────
Agent      : Single Agent
Tools used : get_tickers_by_sector, get_price_performance, get_price_performance
Confidence : 0%
Answer     :
  Here are the top three tech stocks that dropped this month but grew this year:

  1. **Fidelity National Information Services (FIS)**
     - **1-month performance**: +1.78%
     - **Year-to-date performance**: -21.53%

  2. **Jack Henry & Associates, Inc. (JKHY)**
     - **1-month performance**: +0.52%
     - **Year-to-date performance**: -3.31%

  3. **Broadridge Financial Solutions (BR)**
     - **1-month performance**: +8.74%
     - **Year-to-date performance**: -10.70%

  **Note:** 
  The stocks that dropped this
[Single Agent] TOOL CALL REQUESTED → 1 tool(s)
[Single Agent] TOOL → query_local_db {'sql': "SELECT ticker, company FROM stocks WHERE sector = 'Information Technology' AND market_cap = 'Large' AND exchange = 'NASDAQ'"}

──────────────────────────────────────────────────────
Agent   

In [43]:
import pandas as pd
df_logs = pd.DataFrame(logs)
df_logs

,question,answer,score,iterations,elapsed_sec
0,List all semiconductor companies in the database.,Here is a list of semiconductor companies in t...,0.0,2,5.59
1,Are the US stock markets open right now?,The US stock markets are currently closed. The...,0.0,2,2.24
2,What is the P/E ratio of Apple (AAPL)?,The P/E ratio of Apple Inc. (AAPL) is 32.88.,0.0,2,2.23
3,What is the latest news sentiment for Microsof...,The latest news sentiment for Microsoft (MSFT)...,0.0,2,6.96
4,What is NVIDIA's stock price performance over ...,NVIDIA's stock price performance over the last...,0.0,2,2.65
5,"Compare the 1-year price performance of AAPL, ...",Here's the 1-year price performance summary fo...,0.0,5,9.23
6,"Compare the P/E ratios of AAPL, MSFT, and NVDA...",I was able to retrieve the P/E ratio for Apple...,0.0,4,4.56
7,Which energy stocks in the database had the be...,The energy stocks with the best 6-month perfor...,0.0,3,15.82
8,What is the news sentiment for Tesla (TSLA) an...,"For Tesla (TSLA), the news sentiment includes ...",0.0,3,6.00
9,What are the 52-week high and low for JPMorgan...,The 52-week high and low for JPMorgan Chase & ...,0.0,3,4.34


---
## 🛠️ Task 4 — Design and Implement a Multi-Agent System (25 pts)

You must build a multi-agent system that answers the same 15 questions.  
**The architecture is your choice.** Experiment, compare, and justify your decision in the reflections.

---

### Three architectures to consider

**Orchestrator + Specialists + Critic**
```
User Question
     │
 Orchestrator  ← reads question, writes a plan, delegates sub-tasks
     │
 ┌───┼───┐
 Ag1 Ag2 Ag3   ← each handles one domain, sees a subset of tools
 └───┼───┘
  Critic        ← fact-checks each agent's answer vs its raw tool data
     │
 Synthesizer    ← merges verified answers into one final response
```
*Good for:* complex cross-domain questions  
*Tradeoff:* most latency, most API calls, but most transparency

---

**Sequential Pipeline**
```
User Question → Agent1 → Agent2 → Agent3 → Final Answer
```
Each agent receives the previous agent's output as context.  
*Good for:* questions with a natural order (find tickers → get prices → summarise)  
*Tradeoff:* errors propagate — a wrong ticker in step 1 breaks all later steps

---

**Parallel Specialists + Aggregator**
```
User Question
  ├── Price Agent       ─┐
  ├── Fundamentals Agent  ├─→ Aggregator → Final Answer
  └── Sentiment Agent   ─┘
```
All specialists run, aggregator merges results.  
*Good for:* speed (can use `ThreadPoolExecutor` to run in parallel)  
*Tradeoff:* no cross-checking between agents

---

### Suggested tool subsets per specialist
These are starting points — adjust based on your design:

```python
MARKET_TOOLS      = [SCHEMA_TICKERS, SCHEMA_PRICE, SCHEMA_STATUS, SCHEMA_MOVERS]
FUNDAMENTAL_TOOLS = [SCHEMA_OVERVIEW, SCHEMA_SQL, SCHEMA_TICKERS]
SENTIMENT_TOOLS   = [SCHEMA_NEWS, SCHEMA_SQL]
```

Giving each specialist only its relevant schemas reduces wrong tool choices.

---

### Required return contract

Whatever architecture you choose, `run_multi_agent()` must return this dict  
(the evaluation runner reads these fields):

```python
{
    "final_answer"  : str,         # the answer shown to the user
    "agent_results" : list,        # list[AgentResult] — one per specialist that ran
    "elapsed_sec"   : float,       # total wall clock time
    "architecture"  : str,         # short name e.g. "orchestrator-critic", "pipeline", "parallel"
}
```

The evaluation runner extracts from `agent_results`:
- `tools_called` (all tools used across specialists)
- `confidence` (averaged across specialists)
- `issues_found` (all issues concatenated)

---

### Document your design choices in document

The grading for this task includes your reasoning — explain in document:
- Which architecture you chose and why
- What you tried that didn't work
- How you divided tools between specialists
- What your verification/confidence mechanism does


In [72]:
# ── YOUR MULTI-AGENT IMPLEMENTATION ──────────────────────────
#
# Architecture chosen: [describe your choice here]
# Reason: [explain why you chose this over the alternatives]
#
# Specialist breakdown:
#   Agent 1 — [name, domain, tool subset]
#   Agent 2 — [name, domain, tool subset]
#   Agent N — [name, domain, tool subset]
#
# Verification mechanism: [how does your system check answer quality?]
#
### YOUR CODE HERE
def run_multi_agent(question: str) -> AgentResult:
    return []

In [73]:
# Test 1 — check return contract
out1 = run_multi_agent("What is the P/E ratio of Apple (AAPL)?")
assert "final_answer"   in out1, "Missing final_answer key"
assert "agent_results"  in out1, "Missing agent_results key"
assert "elapsed_sec"    in out1, "Missing elapsed_sec key"
assert "architecture"   in out1, "Missing architecture key"
print(f"Architecture : {out1['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out1['agent_results']]}")
print(f"Answer       : {out1['final_answer'][:200]}")

AssertionError: Missing final_answer key

In [ ]:
# Test 2 — cross-domain hard question
out2 = run_multi_agent("For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios and current news sentiment?")
print(f"Architecture : {out2['architecture']}")
print(f"Agents ran   : {[r.agent_name for r in out2['agent_results']]}")
print(f"Tools used   : {[t for r in out2['agent_results'] for t in r.tools_called]}")
print(f"\nAnswer:\n{out2['final_answer'][:400]}")

---
## 🛠️ Task 5 — Implement the LLM Evaluator (15 pts)

The evaluator is an **LLM-as-judge**: it reads a question, the expected answer description, and an agent's actual answer, then scores it.

This is how you measure accuracy across all architectures automatically — rather than reading 45 answers by hand.

---

### Required output format

Your evaluator must return a Python dict with exactly these keys.  
The evaluation runner reads all of them to fill the Excel sheet:

```python
{
    "score"                 : int,        # 0, 1, 2, or 3
    "max_score"             : int,        # always 3
    "reasoning"             : str,        # one sentence explaining the score
    "hallucination_detected": bool,       # True if the answer contains invented facts
    "key_issues"            : list[str],  # specific problems found; empty list if none
}
```

---

### Scoring rubric to include in your prompt

```
3 — Fully correct:    all required data present, numbers accurate, conditions met
2 — Partially correct: key data present but incomplete, gaps, or minor inaccuracies
1 — Mostly wrong:     attempted but wrong numbers, missed required conditions,
                      or claims that appear fabricated
0 — Complete failure: refused to answer, said data unavailable without trying tools,
                      or answer has no relevance to the question
```

---

### Hallucination detection — what to flag

Include these rules in your prompt:
- Specific numbers (prices, P/E ratios, % changes) with no tool data to support them
- Stock tickers that don't exist or aren't relevant to the question
- Definitive claims about "current" data without having called a live data tool

---

### Important considerations

**The evaluator only sees text** — it cannot verify numbers against live data.  
Its hallucination detection is based on reasoning about whether claims look fabricated,  
not on cross-checking against a ground truth source.  
You will reflect on this limitation in the graded questions.

**JSON parsing:** The LLM may wrap its response in markdown fences (```json ... ```).  
Strip those before parsing. Return the fallback dict on any parse error.

**Prompt placement:** Pass the rubric, the expected answer description, and the agent's  
actual answer all in one user message so the evaluator has full context.

---

### Calibration tests (provided below)

Three test cases check your evaluator before the full run:
1. A clearly correct answer (expect score = 3)
2. An answer with an invented number (expect `hallucination_detected = True`, score ≤ 1)
3. A refusal (expect score = 0)

All three must behave as expected before running the full evaluation.


In [74]:
# ── YOUR EVALUATOR IMPLEMENTATION ────────────────────────────
#
# Things to decide:
#   - How detailed is your rubric? (more detail → more consistent scores)
#   - How do you instruct it to detect hallucinations vs honest uncertainty?
#   - How strict are you on partial answers? (affects SA vs MA comparison)
#   - How do you handle "I don't know" answers — 0 or 1?
#
# Required: function signature must be exactly as shown below.

def run_evaluator(question: str, expected_answer: str, agent_answer: str, model="gpt-4o-mini") -> dict:
    """
    Evaluate one agent answer against the expected answer using LLM-as-judge.

    Parameters
    ----------
    question        : str  - original user question
    expected_answer : str  - description of the correct answer
    agent_answer    : str  - text answer produced by the agent
    tools_used      : list - names of tools the agent actually called

    Returns
    -------
    dict with keys:
        score (int, 0-3)
        max_score (3)
        reasoning (str)
        hallucination_detected (bool)
        key_issues (list of str)
    
    On JSON parse failure, returns fallback dict.
    """
    import json
    import re

    # Strip markdown fences if present
    clean_answer = re.sub(r"```[\w]*\n|```", "", agent_answer).strip()

    # System prompt to instruct LLM as judge
    system_prompt = """
You are an expert evaluator scoring answers from AI agents about financial data.
Score the answer 0-3 using the rubric:
3 — Fully correct: all required data present, accurate, conditions met
2 — Partially correct: key data present but incomplete or minor errors
1 — Mostly wrong: wrong numbers, missed conditions, or suspected fabrication
0 — Complete failure: refused, gave up, unrelated answer

Check for hallucinations:
- Numbers, P/E ratios, % changes without tool calls
- Tickers that don't exist
- Claims about 'current' data from agents with no live data
- Fabricated facts

Examples:
- Question: What is the P/E ratio of Apple (AAPL)?
- Answer: The current P/E ratio of Apple Inc. (AAPL) is 33.45.
- Score: 3

Return **only JSON** in the format:

{
    "score": int,
    "max_score": 3,
    "reasoning": str,
    "hallucination_detected": bool,
    "key_issues": list
}
"""

    user_prompt = f"""
Question: {question}
Expected answer description: {expected_answer}
Agent answer: {clean_answer}

Analyze the agent answer. Score it 0-3, detect hallucinations, and list any key issues.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )

        # Parse JSON output safely
        msg_content = response.choices[0].message.content
        # Strip code fences just in case
        msg_content = re.sub(r"```[\w]*\n|```", "", msg_content).strip()
        result = json.loads(msg_content)

        # Ensure keys exist
        required_keys = ["score", "max_score", "reasoning", "hallucination_detected", "key_issues"]
        for k in required_keys:
            if k not in result:
                raise ValueError(f"Missing key {k}")

        # Cast types safely
        result["score"] = int(result["score"])
        result["max_score"] = 3
        result["hallucination_detected"] = bool(result["hallucination_detected"])
        if not isinstance(result["key_issues"], list):
            result["key_issues"] = list(result["key_issues"])

        return result

    except Exception as e:
        # fallback on any parse or API error
        return {
            "score": 0,
            "max_score": 3,
            "reasoning": "evaluator parse error",
            "hallucination_detected": False,
            "key_issues": ["evaluator failed to parse"]
        }


# ── Calibration tests — all three must behave as expected ─────
print("=== Calibration Test 1 — correct answer (expect score=3) ===")
t1 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "The current P/E ratio of Apple Inc. (AAPL) is 33.45.",
)
print(f"  Score: {t1['score']}/3 | Hallucination: {t1['hallucination_detected']}")
print(f"  Reasoning: {t1['reasoning']}")

print("\n=== Calibration Test 2 — fabricated number (expect hallucination=True, score≤1) ===")
t2 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "Apple's P/E ratio is approximately 28.5 based on current market conditions.",
)
print(f"  Score: {t2['score']}/3 | Hallucination: {t2['hallucination_detected']}")
print(f"  Reasoning: {t2['reasoning']}")
assert t2["hallucination_detected"] == True, "Should detect fabricated P/E as hallucination"

print("\n=== Calibration Test 3 — refusal (expect score=0) ===")
t3 = run_evaluator(
    question        = "What is the P/E ratio of Apple (AAPL)?",
    expected_answer = "Should return AAPL P/E ratio as a single numeric value from Alpha Vantage.",
    agent_answer    = "I cannot retrieve real-time financial data. Please check Yahoo Finance.",
)
print(f"  Score: {t3['score']}/3 | Hallucination: {t3['hallucination_detected']}")
print(f"  Reasoning: {t3['reasoning']}")
assert t3["score"] == 0, "Refusal should score 0"

print("\n✅ Evaluator calibration complete")

=== Calibration Test 1 — correct answer (expect score=3) ===
  Score: 3/3 | Hallucination: False
  Reasoning: The answer provides the P/E ratio of Apple Inc. (AAPL) accurately as a single numeric value, which meets the expected answer description.

=== Calibration Test 2 — fabricated number (expect hallucination=True, score≤1) ===
  Score: 1/3 | Hallucination: True
  Reasoning: The answer provides a P/E ratio for Apple (AAPL), but the value is likely inaccurate as it does not reference a specific source or current data, and the agent cannot access live data. The phrase 'based on current market conditions' is vague and does not meet the requirement for a specific numeric value from Alpha Vantage.

=== Calibration Test 3 — refusal (expect score=0) ===
  Score: 0/3 | Hallucination: False
  Reasoning: The agent refused to provide the requested information and directed the user to another source, which does not fulfill the requirement of providing the P/E ratio for Apple (AAPL).

✅ Evaluato

## Step 5 — Benchmark Questions (Fixed — Do Not Modify)

15 questions across three difficulty levels. All three architectures run against all 15.

| Tier | Questions | What makes them harder |
|---|---|---|
| Easy (Q01–Q05) | 5 | 1 tool, single domain |
| Medium (Q06–Q10) | 5 | 2 tools, cross-domain reasoning |
| Hard (Q11–Q15) | 5 | 3+ tools, multi-condition filtering or cross-domain synthesis |


In [75]:
BENCHMARK_QUESTIONS = [
    # ── EASY ──────────────────────────────────────────────────────────────
    {"id":"Q01","complexity":"easy","category":"sector_lookup",
     "question":"List all semiconductor companies in the database.",
     "expected":"Should return company names and tickers for semiconductor stocks from the local DB. "
                "Tickers include NVDA, AMD, INTC, QCOM, AVGO, TXN, ADI, MU and others."},
    {"id":"Q02","complexity":"easy","category":"market_status",
     "question":"Are the US stock markets open right now?",
     "expected":"Should return the current open/closed status for NYSE and NASDAQ "
                "with their trading hours."},
    {"id":"Q03","complexity":"easy","category":"fundamentals",
     "question":"What is the P/E ratio of Apple (AAPL)?",
     "expected":"Should return AAPL P/E ratio as a single numeric value fetched from Alpha Vantage."},
    {"id":"Q04","complexity":"easy","category":"sentiment",
     "question":"What is the latest news sentiment for Microsoft (MSFT)?",
     "expected":"Should return 3–5 recent MSFT headlines with Bullish/Bearish/Neutral labels and scores."},
    {"id":"Q05","complexity":"easy","category":"price",
     "question":"What is NVIDIA's stock price performance over the last month?",
     "expected":"Should return NVDA start price, end price, and % change for the 1-month period."},

    # ── MEDIUM ─────────────────────────────────────────────────────────────
    {"id":"Q06","complexity":"medium","category":"price_comparison",
     "question":"Compare the 1-year price performance of AAPL, MSFT, and GOOGL. Which grew the most?",
     "expected":"Should fetch 1y performance for all 3 tickers, return % change for each, "
                "and identify the highest performer."},
    {"id":"Q07","complexity":"medium","category":"fundamentals",
     "question":"Compare the P/E ratios of AAPL, MSFT, and NVDA. Which looks most expensive?",
     "expected":"Should return P/E ratios for all 3 tickers and identify which has the highest P/E."},
    {"id":"Q08","complexity":"medium","category":"sector_price",
     "question":"Which energy stocks in the database had the best 6-month performance?",
     "expected":"Should query the DB for energy sector tickers, fetch 6-month price performance "
                "for each, and return them ranked by % change."},
    {"id":"Q09","complexity":"medium","category":"sentiment",
     "question":"What is the news sentiment for Tesla (TSLA) and how has its stock moved this month?",
     "expected":"Should return TSLA news sentiment (label + score) AND 1-month price % change "
                "from two separate tool calls."},
    {"id":"Q10","complexity":"medium","category":"fundamentals",
     "question":"What are the 52-week high and low for JPMorgan (JPM) and Goldman Sachs (GS)?",
     "expected":"Should return 52-week high and low for both JPM and GS fetched from Alpha Vantage."},

    # ── HARD ───────────────────────────────────────────────────────────────
    {"id":"Q11","complexity":"hard","category":"multi_condition",
     "question":"Which tech stocks dropped this month but grew this year? Return the top 3.",
     "expected":"Should get tech tickers from DB, fetch both 1-month and year-to-date performance, "
                "filter for negative 1-month AND positive YTD, return top 3 by yearly growth with "
                "exact percentages. Results must satisfy both conditions simultaneously."},
    {"id":"Q12","complexity":"hard","category":"multi_condition",
     "question":"Which large-cap technology stocks on NASDAQ have grown more than 20% this year?",
     "expected":"Should query DB for large-cap NASDAQ tech stocks, fetch YTD performance, "
                "filter for >20% growth, and return matching tickers with exact % change."},
    {"id":"Q13","complexity":"hard","category":"cross_domain",
     "question":"For the top 3 semiconductor stocks by 1-year return, what are their P/E ratios "
                "and current news sentiment?",
     "expected":"Should find semiconductor tickers in DB, rank by 1-year return to find top 3, "
                "then fetch P/E ratio AND news sentiment for each — requiring three separate "
                "data domains (price, fundamentals, sentiment)."},
    {"id":"Q14","complexity":"hard","category":"cross_domain",
     "question":"Compare the market cap, P/E ratio, and 1-year stock performance of JPM, GS, and BAC.",
     "expected":"Should return market cap, P/E, and 1-year % change for all 3 tickers, "
                "combining Alpha Vantage fundamentals and yfinance price data."},
    {"id":"Q15","complexity":"hard","category":"multi_condition",
     "question":"Which finance sector stocks are trading closer to their 52-week low than their "
                "52-week high? Return the news sentiment for each.",
     "expected":"Should get finance sector tickers from DB, fetch 52-week high and low for each, "
                "compute proximity to the low, then fetch news sentiment for qualifying stocks."},
]

print(f"✅ {len(BENCHMARK_QUESTIONS)} questions loaded")
for tier in ["easy","medium","hard"]:
    count = sum(1 for q in BENCHMARK_QUESTIONS if q["complexity"]==tier)
    print(f"   {tier:8s}: {count} questions")

✅ 15 questions loaded
   easy    : 5 questions
   medium  : 5 questions
   hard    : 5 questions


## Step 6 — Evaluation Runner and Excel Writer (Provided)

This runner calls all three architectures on all 15 questions, evaluates each answer,  
and writes results to an Excel file with two sheets:

- **Results** — one row per question with all metrics for all three architectures  
- **Summary** — accuracy % by architecture and difficulty tier (auto-calculated)

Results are saved after every question — if the run crashes, you do not lose progress.

### Excel columns produced

| Column group | Columns written |
|---|---|
| Question | ID, complexity, category, question text, expected answer |
| Baseline | answer, time(s), eval score (0-3), eval reasoning, hallucination detected, issues |
| Single Agent | answer, tools used, tool count, iterations, time(s), eval score, reasoning, hallucination, issues |
| Multi Agent | answer, tools used, tool count, time(s), confidence, critic issues, agents activated, architecture, eval score, reasoning, hallucination, issues |


In [77]:
@dataclass
class EvalRecord:
    # Question
    question_id : str;  question    : str;  complexity : str
    category    : str;  expected    : str
    # Baseline
    bl_answer       : str   = "";  bl_time         : float = 0.0
    bl_score        : int   = -1;  bl_reasoning    : str   = ""
    bl_hallucination: str   = "";  bl_issues       : str   = ""
    # Single agent
    sa_answer       : str   = "";  sa_tools        : str   = ""
    sa_tool_count   : int   = 0;   sa_iters        : int   = 0
    sa_time         : float = 0.0; sa_score        : int   = -1
    sa_reasoning    : str   = "";  sa_hallucination: str   = ""
    sa_issues       : str   = ""
    # Multi agent
    ma_answer       : str   = "";  ma_tools        : str   = ""
    ma_tool_count   : int   = 0;   ma_time         : float = 0.0
    ma_confidence   : str   = "";  ma_critic_issues: int   = 0
    ma_agents       : str   = "";  ma_architecture : str   = ""
    ma_score        : int   = -1;  ma_reasoning    : str   = ""
    ma_hallucination: str   = "";  ma_issues       : str   = ""


# ── Column rename map (internal name → Excel header) ─────────
_COL_NAMES = {
    "question_id":"Question ID","question":"Question","complexity":"Difficulty",
    "category":"Category","expected":"Expected Answer",
    "bl_answer":"Baseline Answer","bl_time":"Baseline Time (s)",
    "bl_score":"Baseline Score /3","bl_reasoning":"Baseline Eval Reasoning",
    "bl_hallucination":"Baseline Hallucination","bl_issues":"Baseline Issues",
    "sa_answer":"SA Answer","sa_tools":"SA Tools Used","sa_tool_count":"SA Tool Count",
    "sa_iters":"SA Iterations","sa_time":"SA Time (s)",
    "sa_score":"SA Score /3","sa_reasoning":"SA Eval Reasoning",
    "sa_hallucination":"SA Hallucination","sa_issues":"SA Issues",
    "ma_answer":"MA Answer","ma_tools":"MA Tools Used","ma_tool_count":"MA Tool Count",
    "ma_time":"MA Time (s)","ma_confidence":"MA Avg Confidence",
    "ma_critic_issues":"MA Critic Issue Count","ma_agents":"MA Agents Activated",
    "ma_architecture":"MA Architecture",
    "ma_score":"MA Score /3","ma_reasoning":"MA Eval Reasoning",
    "ma_hallucination":"MA Hallucination","ma_issues":"MA Issues",
}


def _save_excel(records: list, path: str):
    df = pd.DataFrame([r.__dict__ for r in records]).rename(columns=_COL_NAMES)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # ── Sheet 1: full results ──────────────────────────────
        df.to_excel(writer, index=False, sheet_name="Results")

        # ── Sheet 2: summary by architecture × difficulty ──────
        rows = []
        for arch, sc, tc, hc in [
            ("Baseline",     "Baseline Score /3", "Baseline Time (s)", "Baseline Hallucination"),
            ("Single Agent", "SA Score /3",       "SA Time (s)",       "SA Hallucination"),
            ("Multi Agent",  "MA Score /3",       "MA Time (s)",       "MA Hallucination"),
        ]:
            for tier in ["easy", "medium", "hard", "all"]:
                subset = df if tier == "all" else df[df["Difficulty"] == tier]
                valid  = subset[subset[sc] >= 0]
                avg_s  = valid[sc].mean() if len(valid) else 0
                rows.append({
                    "Architecture"   : arch,
                    "Difficulty"     : tier,
                    "Questions Scored": len(valid),
                    "Avg Score /3"   : round(avg_s, 2),
                    "Accuracy %"     : round(avg_s / 3 * 100, 1),
                    "Avg Time (s)"   : round(df[tc].mean(), 1),
                    "Hallucinations" : (df[hc] == "True").sum(),
                })
        pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="Summary")


def run_full_evaluation(output_xlsx: str = "results.xlsx", delay_sec: float = 3.0):
    """
    Run all 15 questions through baseline, single agent, and multi agent.
    Score each answer. Write results to Excel after every question.
    """
    records = []
    total   = len(BENCHMARK_QUESTIONS)
    print(f"\n{'='*62}")
    print(f"  FULL EVALUATION  |  {total} questions × 3 architectures")
    print(f"  Model: {ACTIVE_MODEL}  |  Output: {output_xlsx}")
    print(f"{'='*62}\n")

    for i, q in enumerate(BENCHMARK_QUESTIONS, 1):
        print(f"[{i:02d}/{total}] {q['id']} ({q['complexity']:6s}) {q['question'][:52]}...")
        rec = EvalRecord(question_id=q["id"], question=q["question"],
                         complexity=q["complexity"], category=q["category"],
                         expected=q["expected"])

        # ── Baseline ───────────────────────────────────────────
        print("         baseline  ...", end=" ", flush=True)
        try:
            t0 = time.time()
            bl = run_baseline(q["question"], verbose=False)
            rec.bl_answer = bl.answer.replace("\n", " ")
            rec.bl_time   = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], bl.answer)
            rec.bl_score        = ev.get("score", -1)
            rec.bl_reasoning    = ev.get("reasoning", "")
            rec.bl_hallucination= str(ev.get("hallucination_detected", False))
            rec.bl_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.bl_time:5.1f}s  score {rec.bl_score}/3")
        except Exception as e:
            print(f"❌  {e}")

        # ── Single Agent ───────────────────────────────────────
        print("         single    ...", end=" ", flush=True)
        try:
            t0 = time.time()
            sa = run_single_agent(q["question"], verbose=False)
            rec.sa_answer    = sa.answer.replace("\n", " ")
            rec.sa_tools     = ", ".join(sa.tools_called)
            rec.sa_tool_count= len(sa.tools_called)
            rec.sa_iters     = len(sa.tools_called) + 1   # approx
            rec.sa_time      = round(time.time() - t0, 2)
            ev = run_evaluator(q["question"], q["expected"], sa.answer)
            rec.sa_score        = ev.get("score", -1)
            rec.sa_reasoning    = ev.get("reasoning", "")
            rec.sa_hallucination= str(ev.get("hallucination_detected", False))
            rec.sa_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.sa_time:5.1f}s  score {rec.sa_score}/3"
                  f"  tools [{rec.sa_tools or 'none'}]")
        except Exception as e:
            print(f"❌  {e}")

        # ── Multi Agent ────────────────────────────────────────
        print("         multi     ...", end=" ", flush=True)
        try:
            t0  = time.time()
            ma  = run_multi_agent(q["question"], verbose=False)
            res = ma.get("agent_results", [])
            all_tools  = [t for r in res for t in r.tools_called]
            all_issues = [iss for r in res for iss in r.issues_found]
            avg_conf   = sum(r.confidence for r in res) / len(res) if res else 0.0
            rec.ma_answer        = ma["final_answer"].replace("\n", " ")
            rec.ma_tools         = ", ".join(dict.fromkeys(all_tools))
            rec.ma_tool_count    = len(all_tools)
            rec.ma_time          = round(time.time() - t0, 2)
            rec.ma_confidence    = f"{avg_conf:.0%}"
            rec.ma_critic_issues = len(all_issues)
            rec.ma_agents        = ", ".join(r.agent_name for r in res)
            rec.ma_architecture  = ma.get("architecture", "")
            ev = run_evaluator(q["question"], q["expected"], ma["final_answer"])
            rec.ma_score        = ev.get("score", -1)
            rec.ma_reasoning    = ev.get("reasoning", "")
            rec.ma_hallucination= str(ev.get("hallucination_detected", False))
            rec.ma_issues       = " | ".join(ev.get("key_issues", []))
            print(f"✅  {rec.ma_time:5.1f}s  score {rec.ma_score}/3"
                  f"  conf {rec.ma_confidence}  issues {rec.ma_critic_issues}")
        except Exception as e:
            print(f"❌  {e}")

        records.append(rec)
        _save_excel(records, output_xlsx)       # save progress after every question

        if i < total:
            print(f"         ⏳ waiting {delay_sec}s ...\n")
            time.sleep(delay_sec)

    # ── Print summary table ────────────────────────────────────
    print(f"\n{'='*62}  RESULTS")
    print(f"{'Architecture':<18} {'Easy':>8} {'Medium':>8} {'Hard':>8} {'Overall':>8}")
    print("─" * 52)
    for arch, sk in [("Baseline","bl_score"),("Single Agent","sa_score"),("Multi Agent","ma_score")]:
        def pct(tier):
            s = [getattr(r, sk) for r in records
                 if getattr(r, sk) >= 0 and (tier=="all" or r.complexity==tier)]
            return f"{sum(s)/len(s)/3*100:.0f}%" if s else "—"
        print(f"{arch:<18} {pct('easy'):>8} {pct('medium'):>8} {pct('hard'):>8} {pct('all'):>8}")

    print(f"\n✅ Saved → {output_xlsx}")
    return output_xlsx

print("✅ Evaluation runner ready")

✅ Evaluation runner ready


## Step 7 — Run the Evaluation

Run the sanity check cell first. Only proceed to the full evaluation once all three architectures  
produce valid output on the test question.


In [79]:
# ── Sanity check — one question, all three architectures ──────
q_test = BENCHMARK_QUESTIONS[2]        # Q03 — easy fundamentals
print(f"Test question: {q_test['question']}\n")

bl_t = run_baseline(q_test["question"], verbose=False)
sa_t = run_single_agent(q_test["question"], verbose=False)
#ma_t = run_multi_agent(q_test["question"], verbose=False)

print(f"Baseline     : {bl_t.answer[:120]}")
print(f"Single Agent : {sa_t.answer[:120]}  |  tools: {sa_t.tools_called}")
#print(f"Multi Agent  : {ma_t['final_answer'][:120]}  |  arch: {ma_t['architecture']}")

ev_bl = run_evaluator(q_test["question"], q_test["expected"], bl_t.answer)
ev_sa = run_evaluator(q_test["question"], q_test["expected"], sa_t.answer)
#ev_ma = run_evaluator(q_test["question"], q_test["expected"], ma_t["final_answer"])
#print(f"\nScores — Baseline: {ev_bl['score']}/3  |  Single: {ev_sa['score']}/3  |  Multi: {ev_ma['score']}/3")

Test question: What is the P/E ratio of Apple (AAPL)?

Baseline     : I'm unable to provide the current P/E ratio of Apple (AAPL) as my data is not up to date. However, you can find the most
Single Agent : The P/E ratio of Apple Inc. (AAPL) is 32.88.  |  tools: ['get_company_overview']


In [80]:
# ── Full evaluation — gpt-4o-mini ─────────────────────────────
ACTIVE_MODEL = MODEL_SMALL
run_full_evaluation(output_xlsx="results_gpt4o_mini.xlsx", delay_sec=3.0)


  FULL EVALUATION  |  15 questions × 3 architectures
  Model: gpt-4o-mini  |  Output: results_gpt4o_mini.xlsx

[01/15] Q01 (easy  ) List all semiconductor companies in the database....
         baseline  ... ✅    1.8s  score 1/3
         single    ... ✅    6.5s  score 2/3  tools [get_tickers_by_sector]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[02/15] Q02 (easy  ) Are the US stock markets open right now?...
         baseline  ... ✅    1.4s  score 2/3
         single    ... ✅    3.1s  score 2/3  tools [get_market_status]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[03/15] Q03 (easy  ) What is the P/E ratio of Apple (AAPL)?...
         baseline  ... ✅    1.5s  score 0/3
         single    ... ✅    1.4s  score 2/3  tools [get_company_overview]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         

$HES: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HES']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


✅   11.7s  score 2/3  tools [get_tickers_by_sector, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[09/15] Q09 (medium) What is the news sentiment for Tesla (TSLA) and how ...
         baseline  ... ✅    2.2s  score 0/3
         single    ... ✅    6.1s  score 2/3  tools [get_news_sentiment, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[10/15] Q10 (medium) What are the 52-week high and low for JPMorgan (JPM)...
         baseline  ... ✅    1.3s  score 0/3
         single    ... ✅    4.2s  score 1/3  tools [get_company_overview, get_company_overview, get_company_overview]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[11/15] Q11 (hard  ) Which tech stocks dropped this month but grew this y...
         baseline  ... ✅    1.3s  score

$FI: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$FI: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")


✅    9.6s  score 0/3  tools [get_tickers_by_sector, get_price_performance, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[12/15] Q12 (hard  ) Which large-cap technology stocks on NASDAQ have gro...
         baseline  ... ✅    1.3s  score 0/3
         single    ... ✅    2.2s  score 0/3  tools [query_local_db]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[13/15] Q13 (hard  ) For the top 3 semiconductor stocks by 1-year return,...
         baseline  ... ✅    2.4s  score 0/3
         single    ... ✅   11.6s  score 1/3  tools [get_tickers_by_sector, get_price_performance, get_company_overview, get_company_overview, get_company_overview, get_news_sentiment, get_news_sentiment, get_news_sentiment]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[14/15] Q14 (h

'results_gpt4o_mini.xlsx'

In [81]:
# ── Full evaluation — gpt-4o (required for reflection Q4) ─────
ACTIVE_MODEL = MODEL_LARGE
run_full_evaluation(output_xlsx="results_gpt4o.xlsx", delay_sec=3.0)


  FULL EVALUATION  |  15 questions × 3 architectures
  Model: gpt-4o  |  Output: results_gpt4o.xlsx

[01/15] Q01 (easy  ) List all semiconductor companies in the database....
         baseline  ... ✅    3.2s  score 1/3
         single    ... ✅   10.8s  score 3/3  tools [get_tickers_by_sector]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[02/15] Q02 (easy  ) Are the US stock markets open right now?...
         baseline  ... ✅    1.5s  score 2/3
         single    ... ✅    2.5s  score 0/3  tools [get_market_status]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[03/15] Q03 (easy  ) What is the P/E ratio of Apple (AAPL)?...
         baseline  ... ✅    1.7s  score 0/3
         single    ... ✅    2.6s  score 0/3  tools [get_company_overview]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 

$HES: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['HES']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


✅   12.4s  score 2/3  tools [get_tickers_by_sector, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[09/15] Q09 (medium) What is the news sentiment for Tesla (TSLA) and how ...
         baseline  ... ✅    2.0s  score 0/3
         single    ... ✅    4.5s  score 1/3  tools [get_news_sentiment, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[10/15] Q10 (medium) What are the 52-week high and low for JPMorgan (JPM)...
         baseline  ... ✅    1.4s  score 0/3
         single    ... ✅    3.9s  score 0/3  tools [get_company_overview, get_company_overview]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[11/15] Q11 (hard  ) Which tech stocks dropped this month but grew this y...
         baseline  ... ✅    1.7s  score 0/3
         single  

$FI: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")
$FI: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")


✅   12.0s  score 0/3  tools [get_tickers_by_sector, get_price_performance, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[12/15] Q12 (hard  ) Which large-cap technology stocks on NASDAQ have gro...
         baseline  ... ✅    1.7s  score 0/3
         single    ... 

$FI: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FI']: possibly delisted; no price data found  (period=ytd) (Yahoo error = "No data found, symbol may be delisted")


✅   10.7s  score 2/3  tools [get_tickers_by_sector, query_local_db, query_local_db, get_price_performance]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[13/15] Q13 (hard  ) For the top 3 semiconductor stocks by 1-year return,...
         baseline  ... ✅    2.4s  score 0/3
         single    ... ✅   12.2s  score 1/3  tools [get_tickers_by_sector, get_price_performance, get_company_overview, get_company_overview, get_company_overview, get_news_sentiment, get_news_sentiment, get_news_sentiment]
         multi     ... ❌  run_multi_agent() got an unexpected keyword argument 'verbose'
         ⏳ waiting 3.0s ...

[14/15] Q14 (hard  ) Compare the market cap, P/E ratio, and 1-year stock ...
         baseline  ... ✅    5.4s  score 1/3
         single    ... ✅    7.8s  score 1/3  tools [get_company_overview, get_company_overview, get_company_overview, get_price_performance]
         multi     ... ❌  run_multi_agent() got an

'results_gpt4o.xlsx'

---
## 📝 Graded Reflection Questions (30 pts)

Answer each question in the markdown cell below it.  
Every answer must reference specific question IDs and scores from your Excel results.  
Vague answers without data will receive at most half marks.


### Q0 — Compare the performance of baseline vs single agent implementation? (5 pts)
- Analyse whether the usecase needs the agentic implementation, if yes why? if no, why not?
**Your answer (minimum 150 words, cite question IDs and scores):**

> From the evaluation (Q01–Q15), the single-agent (SA) implementation performs better than the baseline. The baseline often refused to answer or gave incomplete responses (e.g., Q03 score 0/3, Q06 score 0/3), while the SA tried to use tools and provided more complete answers (e.g., Q03 score 2/3, Q06 score 2/3), even if some numbers were slightly uncertain.
> This shows that for this application, a multi-agent or tool-assisted approach is needed. Many questions require combining data from multiple sources, like stock prices, P/E ratios, and news sentiment. The baseline cannot do this, so it cannot fully answer complex or cross-domain questions (e.g., Q11–Q15). Using a single or multi-agent system allows better coverage, more accurate results, and can handle complex tasks automatically.


### Q1 — Is Multi-Agent Actually Necessary? (5 pts)

Using your `results_gpt4o_mini.xlsx`:

- For which difficulty tier (easy / medium / hard) does multi-agent most outperform single agent?
- For which tier is the difference smallest?
- Give **2 concrete examples** — one where multi-agent clearly won, one where single agent was equivalent or better.  
  For each example, cite the question ID, both scores, and explain *why the architecture made the difference*.

**Your answer (minimum 150 words, cite question IDs and scores):**

> *Replace this text.*


### Q2 — Is the Evaluator Reliable? (5 pts)

Find **3 questions** in your results where you disagree with the score the evaluator assigned.  
For each one:
- What score did it give, and what score do you think is correct?
- Why did it get it wrong — did it miss a hallucination, or penalise an answer that was actually correct?

Then answer: is your evaluator systematically biased in any direction?  
What would you change in your prompt to fix the biggest problem you found?

**Your answer (minimum 150 words):**

> *Replace this text.*


### Q3 — Accuracy Across Architectures and Difficulty Tiers (5 pts)

Fill in the table below from your `results_gpt4o_mini.xlsx` Summary sheet, then write your analysis.

| Architecture | Easy % | Medium % | Hard % | Overall % |
|---|---|---|---|---|
| Baseline | | | | |
| Single Agent | | | | |
| Multi Agent | | | | |

- Where does each architecture most significantly break down?
- Is the accuracy drop from medium → hard larger for some architectures than others?
- What does this tell you about *which type of question* benefits most from an agentic approach?

**Your analysis (minimum 100 words):**

> *Replace this text.*


### Q4 — gpt-4o-mini vs gpt-4o with Your Multi-Agent (5 pts)

Compare `results_gpt4o_mini.xlsx` and `results_gpt4o.xlsx` for the **multi-agent architecture only**.

- Which question categories show the biggest accuracy improvement with the larger model?
- Does the confidence score (or critic issue count) change meaningfully?
- On hard questions specifically — is the accuracy difference large enough to justify the cost?
- Is there any category where the smaller model is actually sufficient?

**Your answer (minimum 150 words):**

> *Replace this text.*


### Q5 — Your Multi-Agent Design Decisions (5 pts)

Document the architecture you built:

1. Which pattern did you choose (orchestrator-critic, pipeline, parallel, or hybrid)? Why?
2. How did you divide tools between specialists? Explain each agent's scope.
3. What is your verification / confidence mechanism and how does it work?
4. What did you try first that did not work, and what did you change?
5. Looking at your results — did your architecture actually reduce hallucinations compared to the single agent? Show the numbers.

**Your answer (minimum 200 words):**

> *Replace this text.*


---
## ✅ Submission Checklist

- [ ] `get_company_overview()` — all assertions in Task 1 pass
- [ ] `get_tickers_by_sector()` — sector match AND industry fallback working
- [ ] `run_baseline()` — `tools_called == []`, answer not empty
- [ ] `run_single_agent()` — uses tools, system prompt reasoning documented in comments
- [ ] `run_multi_agent()` — returns correct dict contract, architecture documented in comments
- [ ] `run_evaluator()` — all 3 calibration tests pass
- [ ] `results_gpt4o_mini.xlsx` — Results + Summary sheets filled for all 15 questions
- [ ] `results_gpt4o.xlsx` — Results + Summary sheets filled for all 15 questions
- [ ] All 5 reflection questions answered with question IDs and scores cited

**Submit:** this notebook + `results_gpt4o_mini.xlsx` + `results_gpt4o.xlsx` + insights.doc/pdf (explaning design choices)
